In [ ]:
import os
import numpy as np
import scipy.io as sio
import pandas as pd

def save_com_distances(base_folder, com_folder_name='COM/predict00'):
    """
    从 base_folder/{com_folder_name}/com3d0.mat 读取 COM 数据，
    计算每帧两只动物的欧式距离，并将结果保存到 CSV（frame, distance）。

    参数
    ----
    base_folder : str
      会话根文件夹，需包含 "{com_folder_name}/com3d0.mat"。
    com_folder_name : str, default 'COM/predict00'
      相对于 base_folder 的 COM 文件子文件夹名。

    效果
    ----
    在 base_folder/{com_folder_name}/ 下生成 com_distances.csv，
    包含两列：frame（帧索引），distance（该帧的欧式距离）。
    """
    com_mat_path = os.path.join(base_folder, com_folder_name, 'com3d0.mat')
    if not os.path.isfile(com_mat_path):
        raise FileNotFoundError(f"未找到 COM 文件: {com_mat_path}")

    mat = sio.loadmat(com_mat_path)
    if 'com' not in mat:
        raise KeyError(f"'com' 未在 {com_mat_path} 中找到")
    raw = mat['com']

    # reshape to (n_frames, 3, n_animals)
    if raw.ndim == 3 and raw.shape[1] == 3:
        com_data = raw
    elif raw.ndim == 2 and raw.shape[1] % 3 == 0:
        n_animals = raw.shape[1] // 3
        com_data = raw.reshape(-1, 3, n_animals)
    else:
        raise ValueError(f"Unexpected COM shape {raw.shape}")
    n_frames, _, n_animals = com_data.shape
    if n_animals < 2:
        raise ValueError("至少需要两只动物才能计算距离")

    # 计算每帧的欧式距离
    a0 = com_data[:, :, 0]
    a1 = com_data[:, :, 1]
    dists = np.linalg.norm(a0 - a1, axis=1)

    # 构造 DataFrame：frame 和 distance
    df = pd.DataFrame({
        'frame':    np.arange(n_frames),
        'distance': dists
    })

    # 保存 CSV
    output_dir = os.path.join(base_folder, com_folder_name)
    os.makedirs(output_dir, exist_ok=True)
    csv_path = os.path.join(output_dir, 'com_distances.csv')
    df.to_csv(csv_path, index=False)
    print(f"已将 COM 距离保存到：{csv_path}")

    return df

session_paths = [
    
    # "/data/big_rim/rsync_dcc_sum/Oct3V1/2024_12_18/20240919v1l5r1mini_p20240717PMC_social_test_11_30",
    # "/data/big_rim/rsync_dcc_sum/Oct3V1/2025_02_27/20241015PMCBE1mini_p20241015PMCRE1_12_33",
    # "/data/big_rim/rsync_dcc_sum/Oct3V1/2024_12_31/20240919v1l5r2mini_p20240717PMC_social_14_04",
    # "/data/big_rim/rsync_dcc_sum/Oct3V1/2024_11_01/2social_mini_20240910V1r_AO_single_12_50",
    # "/data/big_rim/rsync_dcc_sum/Oct3V1/2024_11_01/2social_mini_20240819V1r1_AO_single_14_30",
    # "/data/big_rim/rsync_dcc_sum/Oct3V1/2024_10_31/2social_mini_20240819V1r1_single_11_29",
    # "/data/big_rim/rsync_dcc_sum/Oct3V1/2024_10_31/2social_mini_20240819V1r1_femalebleach_11_48",
    # "/data/big_rim/rsync_dcc_sum/Oct3V1/2025_05_16/20241216V1RE1Fmini_p20241216RE2",
    # "/data/big_rim/rsync_dcc_sum/Oct3V1/2025_05_16/20241216V1RE1Fmini_p20241224PMCLE1",
    # "/data/big_rim/rsync_dcc_sum/Oct3V1/2025_05_16/20240303PMCBE0r1coatedmini_p20240303RE1"
    ]


for ss in session_paths:
    save_com_distances(base_folder=ss)

# # ========== 示例调用 ==========
# if __name__ == '__main__':
#     session_folder = "/data/big_rim/rsync_dcc_sum/Oct3V1/2024_10_31/2social_mini_20240819V1r1_femalebleach_11_48/"  # 替换为实际路径
#     save_com_distances(base_folder=session_folder)


ValueError: 至少需要两只动物才能计算距离

In [ ]:
import os
import json
import pandas as pd

def filter_com_by_mapped_frames(
    base_folder,
    com_folder_name='COM/predict00',
    aligned_folder_name='MIR_Aligned',
    frame_mapping_filename='frame_mapping.json',
    output_filename='com_distances_filtered.csv'
):
    """
    从 base_folder/{com_folder_name}/com_distances.csv 读取全部帧的 COM 距离，
    然后根据 base_folder/{aligned_folder_name}/{frame_mapping_filename} 中的
    "mapped_sixcam_frame_indices" 列表，只保留这些帧对应的距离，
    并将结果保存到 base_folder/{aligned_folder_name}/{output_filename}。

    参数
    ----
    base_folder : str
      会话根文件夹路径。
    com_folder_name : str, default 'COM/predict00'
      COM 数据所在子文件夹名，即之前保存 com_distances.csv 的位置。
    aligned_folder_name : str, default 'MIR_Aligned'
      包含 frame_mapping.json 的文件夹名。
    frame_mapping_filename : str, default 'frame_mapping.json'
      存放过滤帧索引的 JSON 文件名，内部需包含键 "mapped_sixcam_frame_indices"。
    output_filename : str, default 'com_distances_filtered.csv'
      过滤后保存结果的文件名。

    返回
    ----
    pd.DataFrame
      仅包含映射帧和对应距离的 DataFrame。
    """
    # 构建 com_distances.csv 的完整路径
    com_csv_path = os.path.join(base_folder, com_folder_name, 'com_distances.csv')
    if not os.path.isfile(com_csv_path):
        raise FileNotFoundError(f"未找到 COM 距离文件: {com_csv_path}")

    # 构建 frame_mapping.json 的完整路径
    mapping_json_path = os.path.join(base_folder, aligned_folder_name, frame_mapping_filename)
    if not os.path.isfile(mapping_json_path):
        raise FileNotFoundError(f"未找到帧映射文件: {mapping_json_path}")

    # 1. 读取 COM 距离 CSV
    df_com = pd.read_csv(com_csv_path)
    if 'frame' not in df_com.columns or 'distance' not in df_com.columns:
        raise KeyError(f"{com_csv_path} 中必须包含 'frame' 与 'distance' 列")

    # 2. 读取 JSON 并提取映射帧索引列表
    with open(mapping_json_path, 'r') as f:
        mapping_data = json.load(f)

    if 'mapped_sixcam_frame_indices' not in mapping_data:
        raise KeyError(f"JSON 文件中缺少键 'mapped_sixcam_frame_indices'")

    mapped_frames = mapping_data['mapped_sixcam_frame_indices']
    if not isinstance(mapped_frames, list):
        raise ValueError(f"'mapped_sixcam_frame_indices' 应为列表")

    # 3. 过滤 DataFrame，只保留 frame 在 mapped_frames 中的行
    df_filtered = df_com[df_com['frame'].isin(mapped_frames)].copy()

    # 4. 确保输出目录存在
    output_dir = os.path.join(base_folder, aligned_folder_name)
    os.makedirs(output_dir, exist_ok=True)

    # 5. 保存过滤后的 CSV
    output_path = os.path.join(output_dir, output_filename)
    df_filtered.to_csv(output_path, index=False)
    print(f"已将过滤后的 COM 距离保存到：{output_path}")

    return df_filtered
for ss in session_paths:
    filter_com_by_mapped_frames(base_folder=ss)
# # ========== 示例调用 ==========
# if __name__ == '__main__':
#     session_folder = "/data/big_rim/rsync_dcc_sum/Oct3V1/2024_10_31/2social_mini_20240819V1r1_femalebleach_11_48/"  # 替换为实际路径
#     df_result = filter_com_by_mapped_frames(base_folder=session_folder)
